# 🧠 Veri Madenciliği — Sınıflandırma: İleri Düzey Teknikler II
## 3. Ders Uygulamalı Notebook
**Haydar Kılıç | Yapay Zeka Mühendisliği | Bahar Yarıyılı**

---
### 📚 Bu Notebook'un İçeriği

| # | Bölüm | Konu |
|---|-------|------|
| 1 | ANN Derinlemesine | Biyolojik Nöron, İleri/Geri Yayılım, Mini-batch |
| 2 | SVM | Lineer SVM, Dual Problem, Kernel Trick |
| 3 | Bayes Ağları | DAG, Koşullu Bağımsızlık, CPT |
| 4 | Topluluk Öğrenmesi | Bagging, AdaBoost, Bias-Variance |
| 5 | Sınıf Dengesizliği | Undersampling, Oversampling, SMOTE |
| 6 | Performans Metrikleri | Confusion Matrix, ROC, PR Eğrisi |

In [ ]:
# ============================================================
# KURULUM VE İMPORTLAR
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, roc_curve, auc,
                              precision_recall_curve, f1_score,
                              precision_score, recall_score)
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer
from scipy.special import expit as sigmoid
from scipy.stats import norm

print('✅ Tüm kütüphaneler başarıyla yüklendi!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

---
# BÖLÜM 1: Yapay Sinir Ağları (ANN) — Derinlemesine

## 📖 Biyolojik → Yapay Nöron Analogisi

| Biyolojik | Yapay Karşılık |
|-----------|----------------|
| Dendrit | Girdi (x₁, x₂, ..., xₙ) |
| Sinaps gücü | Ağırlık (w₁, w₂, ..., wₙ) |
| Hücre gövdesi (soma) | Toplama fonksiyonu (Σ) |
| Akson | Aktivasyon çıktısı |
| Ateşlenme eşiği | Bias (b) + Aktivasyon fonksiyonu |

## Tek Nöron Matematik
$$z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^T\mathbf{x} + b$$
$$\hat{y} = f(z) \quad \text{(aktivasyon fonksiyonu)}$$

## Çok Katmanlı Ağ — Katman l, Nöron i
$$a_i^l = f(z_i^l) = f\left(\sum_j w_{ij}^l a_j^{l-1} + b_i^l\right)$$

In [ ]:
# ============================================================
# TEK NÖRON: İLERİ YAYILIM ADIM ADIM
# ============================================================
print('=' * 60)
print('TEK NÖRON — İLERİ YAYILIM SİMÜLASYONU')
print('=' * 60)

# Parametreler
x = np.array([2.0, 3.0, -1.0])  # girdiler
w = np.array([0.4, -0.2, 0.7])  # ağırlıklar
b = 0.1                           # bias

# Toplama
z = np.dot(w, x) + b
print(f'\nGirdiler    : x = {x}')
print(f'Ağırlıklar  : w = {w}')
print(f'Bias        : b = {b}')
print(f'\nToplama  z  = w₁x₁ + w₂x₂ + w₃x₃ + b')
print(f'           = ({w[0]}×{x[0]}) + ({w[1]}×{x[1]}) + ({w[2]}×{x[2]}) + {b}')
print(f'           = {w[0]*x[0]:.1f} + {w[1]*x[1]:.1f} + {w[2]*x[2]:.1f} + {b}')
print(f'           = {z:.4f}')

# Farklı aktivasyon fonksiyonları
aktivasyonlar = {
    'Sigmoid σ(z)' : 1 / (1 + np.exp(-z)),
    'Tanh'         : np.tanh(z),
    'ReLU'         : max(0, z),
    'Sign (±1)'    : np.sign(z)
}
print("{:<20} {}".format("Aktivasyon", "Çıktı"))
print('-' * 35)
for isim, deger in aktivasyonlar.items():
    print(f'{isim:<20} {deger:.6f}')

## 1.1 Slayttaki Mini-batch Örneği — Adım Adım

**Model:** Tek nöron → $z = wx + b$, $\hat{y} = \sigma(z)$

**Kayıp:** $\text{Loss}_k = \frac{1}{2}(y_k - \hat{y}_k)^2$

**Başlangıç:** $w^{(0)} = 0.5$, $b^{(0)} = 0$, $\eta = 0.1$

**Mini-batch:** $(x_1,y_1)=(0,0)$, $(x_2,y_2)=(1,1)$, $(x_3,y_3)=(2,1)$

In [ ]:
# ============================================================
# SLAYTTAKI MİNİ-BATCH ÖRNEĞİ — TAM UYGULAMA
# ============================================================
def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

# Veri ve başlangıç
veri = [(0, 0), (1, 1), (2, 1)]
w, b = 0.5, 0.0
eta  = 0.1

print('=' * 65)
print('MİNİ-BATCH EĞİTİMİ — SLAYTTA VERİLEN ÖRNEK')
print('=' * 65)
print(f'Başlangıç: w={w}, b={b}, η={eta}')

# === İLERİ YAYILIM ===
print('\n📌 ADIM 1: İLERİ YAYILIM')
print("{:<5} {:<6} {:<6} {:<14} {:<14} {}".format("k", "xₖ", "yₖ", "zₖ=wxₖ+b", "ŷₖ=σ(zₖ)", "Lossₖ"))
print('-' * 60)

tahminler = []
kayiplar  = []
for k, (x_k, y_k) in enumerate(veri, 1):
    z_k    = w * x_k + b
    yhat_k = sigmoid_fn(z_k)
    loss_k = 0.5 * (y_k - yhat_k) ** 2
    tahminler.append(yhat_k)
    kayiplar.append(loss_k)
    print(f'{k:<5} {x_k:<6} {y_k:<6} {z_k:<14.4f} {yhat_k:<14.4f} {loss_k:.4f}')

print(f'\n  Mini-batch Toplam Kayıp = {sum(kayiplar):.4f}')
print(f'  (Slaytta: ≈ 0.232 ✅)')

# === GERİ YAYILIM ===
print('\n📌 ADIM 2: GERİ YAYILIM (Gradyan Hesabı)')
print('   Sigmoid türevi: σ\'(z) = ŷ(1-ŷ)')
print('   ∂Loss/∂w = (ŷ-y)·σ\'(z)·x    |    ∂Loss/∂b = (ŷ-y)·σ\'(z)')
print()
print("{:<5} {:<9} {:<12} {:<12} {}".format("k", "ŷₖ", "σ'(zₖ)", "∂L/∂b", "∂L/∂w"))
print('-' * 55)

dL_dw_toplam = 0.0
dL_db_toplam = 0.0
for k, (x_k, y_k) in enumerate(veri, 1):
    yhat_k    = tahminler[k-1]
    z_k       = w * x_k + b
    sigma_prime = yhat_k * (1 - yhat_k)
    dL_db_k   = (yhat_k - y_k) * sigma_prime
    dL_dw_k   = dL_db_k * x_k
    dL_dw_toplam += dL_dw_k
    dL_db_toplam += dL_db_k
    print(f'{k:<5} {yhat_k:<9.4f} {sigma_prime:<12.4f} {dL_db_k:<12.4f} {dL_dw_k:.4f}')

print(f'\n  ΔW = {dL_dw_toplam:.4f}  (Slayta göre: ≈ -0.195)')
print(f'  Δb = {dL_db_toplam:.4f}  (Slayta göre: ≈ -0.017)')

# === GÜNCELLEME ===
print('\n📌 ADIM 3: PARAMETRE GÜNCELLEME (Gradient Descent)')
w_yeni = w - eta * dL_dw_toplam
b_yeni = b - eta * dL_db_toplam
print(f'  w_yeni = {w} - {eta}×({dL_dw_toplam:.4f}) = {w_yeni:.4f}')
print(f'  b_yeni = {b} - {eta}×({dL_db_toplam:.4f}) = {b_yeni:.4f}')
print(f'\n  (w, b): ({w}, {b}) ──▶ ({w_yeni:.4f}, {b_yeni:.4f})')
print(f'  Slaytta: (0.5, 0) → (0.5195, 0.0017) ✅')

In [ ]:
# ============================================================
# GERİ YAYILIM — KAYIP VE GRADYAN GÖRSELLEŞTİRME
# ============================================================

# Birden fazla epoch simülasyonu
w_sim, b_sim = 0.5, 0.0
eta_sim = 0.1
n_epoch = 200

kayip_gecmisi = []
w_gecmisi, b_gecmisi = [w_sim], [b_sim]

for epoch in range(n_epoch):
    dw_total = 0.0
    db_total = 0.0
    toplam_kayip = 0.0
    for x_k, y_k in veri:
        z_k      = w_sim * x_k + b_sim
        yhat_k   = sigmoid_fn(z_k)
        sp       = yhat_k * (1 - yhat_k)
        toplam_kayip += 0.5 * (y_k - yhat_k)**2
        dw_total += (yhat_k - y_k) * sp * x_k
        db_total += (yhat_k - y_k) * sp
    w_sim -= eta_sim * dw_total
    b_sim -= eta_sim * db_total
    kayip_gecmisi.append(toplam_kayip)
    w_gecmisi.append(w_sim)
    b_gecmisi.append(b_sim)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Mini-batch Eğitimi: Öğrenme Süreci Görselleştirmesi', fontsize=13, fontweight='bold')

# Kayıp eğrisi
axes[0].plot(kayip_gecmisi, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Toplam Kayıp')
axes[0].set_title('Kayıp Eğrisi'); axes[0].grid(True, alpha=0.3)
axes[0].axhline(kayip_gecmisi[-1], color='red', linestyle='--', 
                label=f'Son kayıp: {kayip_gecmisi[-1]:.4f}')
axes[0].legend()

# Ağırlık değişimi
axes[1].plot(w_gecmisi, 'g-', linewidth=2, label='w')
axes[1].plot(b_gecmisi, 'r-', linewidth=2, label='b')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Parametre Değeri')
axes[1].set_title('Parametre Değişimi'); axes[1].grid(True, alpha=0.3)
axes[1].legend(); axes[1].axvline(0, color='gray', alpha=0.5)

# Tahmin vs gerçek (son model)
x_plot = np.linspace(-0.5, 2.5, 100)
y_plot = sigmoid_fn(w_sim * x_plot + b_sim)
axes[2].plot(x_plot, y_plot, 'b-', linewidth=2, label='Model Tahmini')
for x_k, y_k in veri:
    axes[2].scatter(x_k, y_k, c='red', s=100, zorder=5)
axes[2].scatter([], [], c='red', s=100, label='Gerçek Etiket')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Eşik=0.5')
axes[2].set_xlabel('x'); axes[2].set_ylabel('σ(wx+b)')
axes[2].set_title(f'Son Model (w={w_sim:.3f}, b={b_sim:.3f})')
axes[2].grid(True, alpha=0.3); axes[2].legend()

plt.tight_layout()
plt.show()
print(f'\n✅ {n_epoch} epoch sonrası: w={w_sim:.4f}, b={b_sim:.4f}')

In [ ]:
# ============================================================
# ÇOK KATMANLI AĞ — İLERİ YAYILIM SIFIRDAN
# Mimari: 2 → 3 → 1  (Girdi → Gizli → Çıktı)
# ============================================================

np.random.seed(42)

# Ağırlıklar (rastgele başlat)
W1 = np.random.randn(3, 2) * 0.5   # Gizli katman: 3 nöron, 2 girdi
b1 = np.zeros((3, 1))
W2 = np.random.randn(1, 3) * 0.5   # Çıktı katmanı: 1 nöron, 3 girdi
b2 = np.zeros((1, 1))

def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

# Örnek girdi
x_input = np.array([[1.5], [-0.8]])  # (2, 1)

print('=== ÇOK KATMANLI AĞ: İLERİ YAYILIM (2→3→1) ===')
print(f'Girdi x = {x_input.flatten()}')
print(f'\nW1 (3×2):\n{W1.round(3)}')
print(f'b1 (3×1): {b1.flatten()}')

# Katman 1 (Gizli)
z1 = W1 @ x_input + b1                # (3,1)
a1 = sigmoid(z1)                       # (3,1)
print(f'\n📌 Gizli Katman (Layer 1):')
print(f'  z1 = W1·x + b1 = {z1.flatten().round(4)}')
print(f'  a1 = σ(z1)     = {a1.flatten().round(4)}')

# Katman 2 (Çıktı)
z2 = W2 @ a1 + b2                     # (1,1)
a2 = sigmoid(z2)                       # (1,1)
print(f'\n📌 Çıktı Katmanı (Layer 2):')
print(f'  z2 = W2·a1 + b2 = {z2.flatten().round(4)}')
print(f'  a2 = σ(z2)      = {a2.flatten().round(4)}')
print(f'\n🎯 Tahmin: ŷ = {a2[0,0]:.4f} → Sınıf: {"+1" if a2[0,0]>0.5 else "-1"}')

# Ağ yapısı görsel
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(0, 4); ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Çok Katmanlı Sinir Ağı: 2→3→1', fontsize=13, fontweight='bold')

katmanlar = {
    'Girdi (Layer 0)': ([0.5], [1.5, 3.5], 'lightblue'),
    'Gizli (Layer 1)': ([2.0], [1.0, 2.5, 4.0], 'lightyellow'),
    'Çıktı (Layer 2)': ([3.5], [2.5], 'lightgreen')
}

pozisyonlar = {}
for kat_adi, (xs, ys, renk) in katmanlar.items():
    x_pos = xs[0]
    for i, y_pos in enumerate(ys):
        cercle = plt.Circle((x_pos, y_pos), 0.3, color=renk, ec='black', linewidth=1.5, zorder=5)
        ax.add_patch(cercle)
        etk = kat_adi.split()[0]
        if etk == 'Girdi':
            ax.text(x_pos, y_pos, f'x{i+1}', ha='center', va='center', fontsize=9, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{x_input[i,0]:.1f}', ha='center', va='top', fontsize=7, color='navy')
        elif etk == 'Gizli':
            ax.text(x_pos, y_pos, f'h{i+1}', ha='center', va='center', fontsize=9, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{a1[i,0]:.3f}', ha='center', va='top', fontsize=7, color='darkred')
        else:
            ax.text(x_pos, y_pos, 'ŷ', ha='center', va='center', fontsize=11, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{a2[0,0]:.3f}', ha='center', va='top', fontsize=7, color='darkgreen')
        pozisyonlar[(kat_adi, i)] = (x_pos, y_pos)

# Bağlantılar (girdi→gizli)
for i in range(2):
    for j in range(3):
        x0, y0 = 0.5, [1.5, 3.5][i]
        x1, y1 = 2.0, [1.0, 2.5, 4.0][j]
        ax.annotate('', xy=(x1-0.3, y1), xytext=(x0+0.3, y0),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# Bağlantılar (gizli→çıktı)
for j in range(3):
    x0, y0 = 2.0, [1.0, 2.5, 4.0][j]
    ax.annotate('', xy=(3.2, 2.5), xytext=(x0+0.3, y0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

for x_pos, label in [(0.5, 'Girdi\nKatmanı'), (2.0, 'Gizli\nKatman'), (3.5, 'Çıktı\nKatmanı')]:
    ax.text(x_pos, 0.1, label, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.text(1.1, 4.5, 'Sigmoid aktivasyon →', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SKLEARN MLP — FARKLI MİMARİLER KARŞILAŞTIRMASI
# ============================================================
from sklearn.datasets import load_iris
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print('=== MLP MİMARİ KARŞILAŞTIRMASI (Iris Veri Seti) ===')
print("\n{:<30} {:<12} {:<8} {}".format("Mimari", "Aktivasyon", "Epoch", "Test Doğruluk"))
print('─' * 65)

sonuclar_mlp = []
for mimari in [(10,), (50,), (100,), (50, 25), (100, 50), (100, 50, 25)]:
    for aktivasyon in ['relu', 'tanh', 'logistic']:
        mlp = MLPClassifier(hidden_layer_sizes=mimari, activation=aktivasyon,
                            max_iter=1000, random_state=42)
        mlp.fit(X_tr_s, y_tr)
        acc = accuracy_score(y_te, mlp.predict(X_te_s))
        sonuclar_mlp.append({'Mimari': str(mimari), 'Aktivasyon': aktivasyon,
                             'Epoch': mlp.n_iter_, 'Doğruluk': acc})

df_mlp = pd.DataFrame(sonuclar_mlp)

# En iyi 5 kombinasyonu göster
top5 = df_mlp.nlargest(5, 'Doğruluk')
for _, row in top5.iterrows():
    print(f"{row['Mimari']:<30} {row['Aktivasyon']:<12} {int(row['Epoch']):<8} {row['Doğruluk']:.4f}")

print(f'\n💡 Test örnekleri: {len(y_te)} | Sınıf sayısı: 3 (Iris türleri)')

# Görselleştir
fig, ax = plt.subplots(figsize=(10, 5))
for akt in ['relu', 'tanh', 'logistic']:
    df_f = df_mlp[df_mlp['Aktivasyon'] == akt].reset_index(drop=True)
    ax.plot(range(len(df_f)), df_f['Doğruluk'], 'o-', label=akt, linewidth=2, markersize=8)

ax.set_xticks(range(6))
ax.set_xticklabels(['(10,)', '(50,)', '(100,)', '(50,25)', '(100,50)', '(100,50,25)'], rotation=20)
ax.set_ylabel('Test Doğruluğu'); ax.set_xlabel('Mimari')
ax.set_title('MLP: Mimari × Aktivasyon Karşılaştırması', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.show()

---
# BÖLÜM 2: Destek Vektör Makineleri (SVM)

## 📖 Teorik Özet

**Primal Problem:** $\min_{w,b} \frac{\|w\|^2}{2}$ koşuluyla $y_i(w^T x_i + b) \geq 1$

**Dual Problem:** $\max_\lambda \sum_i \lambda_i - \frac{1}{2}\sum_{i,j} \lambda_i \lambda_j y_i y_j x_i^T x_j$

**Marj Genişliği:** $\frac{2}{\|w\|}$

**KKT Koşulu:** $\lambda_i[y_i(w^T x_i + b) - 1] = 0$ → λ > 0 olanlar Destek Vektörler

## 2.1 Slayttaki SVM Örneği — Manuel Çözüm

**Veri:**
- $x_1=(2,2), y_1=+1$ → $x_2=(4,4), y_2=+1$
- $x_3=(2,0), y_3=-1$ → $x_4=(0,2), y_4=-1$

**Çözüm:** $w=(1,1)$, $b=-3$, Karar sınırı: $x_1+x_2=3$, Marj: $\sqrt{2}$

In [ ]:
# ============================================================
# SLAYTTA VERİLEN SVM ÖRNEĞİ — MANUEL DOĞRULAMA
# ============================================================
from sklearn.svm import SVC

# Slayttaki 4 noktalı örnek
X_svm4 = np.array([[2,2],[4,4],[2,0],[0,2]], dtype=float)
y_svm4 = np.array([1, 1, -1, -1])

print('=== SLAYTTA VERİLEN SVM ÖRNEĞİ ===')
print('Eğitim Veri Kümesi:')
for xi, yi in zip(X_svm4, y_svm4):
    print(f'  x={xi}, y={yi:+d}')

# sklearn ile çöz
svm4 = SVC(kernel='linear', C=1e6)
svm4.fit(X_svm4, y_svm4)

w_sol = svm4.coef_[0]
b_sol = svm4.intercept_[0]
marj  = 2.0 / np.linalg.norm(w_sol)

print(f'\n📐 ÇÖZÜM:')
print(f'  w = [{w_sol[0]:.4f}, {w_sol[1]:.4f}]   (Slaytta: [1, 1])')
print(f'  b = {b_sol:.4f}              (Slaytta: -3)')
print(f'  Marj genişliği = 2/||w|| = {marj:.4f}  (Slaytta: √2 ≈ 1.4142)')
print(f'  Karar sınırı: {w_sol[0]:.2f}·x₁ + {w_sol[1]:.2f}·x₂ + {b_sol:.2f} = 0')
print(f'  → x₁ + x₂ = 3 ✅')

print(f'\n🔵 Destek Vektörleri:')
for sv in svm4.support_vectors_:
    f_val = w_sol @ sv + b_sol
    print(f'  {sv} → f(x)={f_val:.4f}')

# KKT koşullarını kontrol et
print(f'\n✅ KKT Koşulları (λᵢ[yᵢ(w·xᵢ+b) - 1] = 0):')
for xi, yi in zip(X_svm4, y_svm4):
    fonk = yi * (w_sol @ xi + b_sol) - 1
    durum = 'Destek Vektör ✓' if abs(fonk) < 0.01 else 'İç nokta'
    print(f'  x={xi}, y={yi:+d}: yᵢ(w·xᵢ+b)-1 = {fonk:.4f} → {durum}')

In [ ]:
# ============================================================
# SVM GÖRSELLEŞTİRME — MARJ VE DESTEK VEKTÖRLER
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SVM: Maksimum Marjlı Hiperdüzlem', fontsize=13, fontweight='bold')

# Sol: 4 noktalı örnek
ax = axes[0]
x1_r = np.linspace(-0.5, 5, 200)

def svm_line(x1, w, b, offset=0):
    return (-w[0]*x1 - b + offset) / w[1]

ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol),     'k-',  lw=2.5, label='Karar Sınırı')
ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol,  1),  'b--', lw=1.5, label='Marj +1')
ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol, -1),  'r--', lw=1.5, label='Marj -1')

y_up   = svm_line(x1_r, w_sol, b_sol, 1)
y_down = svm_line(x1_r, w_sol, b_sol, -1)
ax.fill_between(x1_r, y_down, y_up, alpha=0.1, color='yellow', label='Marj Bölgesi')

for xi, yi in zip(X_svm4, y_svm4):
    c = '#2ecc71' if yi==1 else '#e74c3c'
    m = 's' if yi==1 else 'o'
    ax.scatter(*xi, c=c, marker=m, s=150, edgecolors='black', zorder=5)

for sv in svm4.support_vectors_:
    ax.add_patch(plt.Circle(sv, 0.25, color='blue', fill=False, lw=2.5, zorder=6))

ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Slaytta Verilen 4 Noktalı Örnek')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Marj genişliği oku
ax.annotate('', xy=(1.5, 1.5), xytext=(2.5, 0.5),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text(1.5, 0.8, f'Marj={marj:.3f}\n(=√2)', color='purple', fontsize=8)

# Sağ: Büyük veri seti
ax2 = axes[1]
np.random.seed(42)
X_buyuk = np.r_[np.random.randn(30, 2) + [2, 2], np.random.randn(30, 2) + [-2, -2]]
y_buyuk = np.r_[np.ones(30), -np.ones(30)]

svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_buyuk, y_buyuk)

xx, yy = np.meshgrid(np.linspace(-5, 6, 200), np.linspace(-5, 6, 200))
Z = svm_lin.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z, alpha=0.25, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
ax2.contour(xx, yy, Z, colors='black', linewidths=1)

for xi, yi in zip(X_buyuk, y_buyuk):
    c = '#27ae60' if yi==1 else '#c0392b'
    ax2.scatter(*xi, c=c, s=40, alpha=0.7)
for sv in svm_lin.support_vectors_:
    ax2.add_patch(plt.Circle(sv, 0.2, color='blue', fill=False, lw=2, zorder=6))

ax2.set_xlim(-5, 6); ax2.set_ylim(-5, 6)
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
ax2.set_title(f'Büyük Veri Seti (Destek Vektör sayısı: {len(svm_lin.support_vectors_)})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KERNEL TRICK — ÖZELLİK UZAYI DÖNÜŞÜMÜ
# ============================================================
# Slayttaki örnek: φ: (x1,x2) → (x1²-x1, x2²-x2)

np.random.seed(42)
X_nl, y_nl = make_circles(n_samples=150, noise=0.12, factor=0.45, random_state=42)
y_nl_svm   = np.where(y_nl==0, -1, 1)

# Orijinal uzayda dönüşüm uygula
def phi_transform(X):
    return np.column_stack([X[:,0]**2 - X[:,0],
                            X[:,1]**2 - X[:,1]])

X_phi = phi_transform(X_nl)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Kernel Trick: Doğrusal Olmayan Dönüşüm', fontsize=13, fontweight='bold')

renkler = ['#c0392b' if yi==-1 else '#27ae60' for yi in y_nl_svm]

# 1. Orijinal uzay (çözülemez)
ax = axes[0]
ax.scatter(X_nl[:,0], X_nl[:,1], c=renkler, s=40, alpha=0.8)
ax.set_title('Özellik Uzayı (Çözümsüz ❌)', fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.grid(True, alpha=0.3)

# 2. Dönüştürülmüş uzay
ax = axes[1]
ax.scatter(X_phi[:,0], X_phi[:,1], c=renkler, s=40, alpha=0.8)
ax.set_title('φ(X) Uzayı: (x₁²-x₁, x₂²-x₂)', fontweight='bold')
ax.set_xlabel('x₁²-x₁'); ax.set_ylabel('x₂²-x₂'); ax.grid(True, alpha=0.3)
ax.set_xlim(-0.3, 0.3); ax.set_ylim(-0.3, 0.3)

# 3. Kernel SVM (RBF) — sınır
ax = axes[2]
svm_rbf = SVC(kernel='rbf', gamma=2.0, C=1.0)
svm_rbf.fit(X_nl, y_nl_svm)
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
Z = svm_rbf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
ax.contour(xx, yy, Z, colors='black', linewidths=1)
ax.scatter(X_nl[:,0], X_nl[:,1], c=renkler, s=40, alpha=0.8)
for sv in svm_rbf.support_vectors_:
    ax.add_patch(plt.Circle(sv, 0.07, color='blue', fill=False, lw=1.5, zorder=6))
acc_rbf = accuracy_score(y_nl_svm, svm_rbf.predict(X_nl))
ax.set_title(f'RBF Kernel SVM (Doğruluk: {acc_rbf:.1%}) ✅', fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 Kernel Trick: φ(x) hesaplamadan K(u,v) = <φ(u),φ(v)> kullan!')
print('   → Polinom: K(u,v) = (uᵀv+1)ᵖ')
print('   → RBF    : K(u,v) = exp(-||u-v||²/2σ²)')
print('   → Sigmoid: K(u,v) = tanh(kuᵀv - δ)')

---
# BÖLÜM 3: Bayes Ağları

## 📖 Teorik Özet

**DAG:** Yönlü Çevrimsiz Çizge — her düğüm bir rastgele değişken

**Yerel Markov Özelliği:**
$$X_i \perp \text{NonDesc}(X_i) \mid pa(X_i)$$

**Ortak Dağılım (Zincir Kuralı):**
$$P(X) = \prod_{i=1}^{d} P(X_i \mid pa(X_i))$$

**Kalp Hastalığı Örneği (Slaytta):**
$$P(X) = P(Eg)\cdot P(Di)\cdot P(KH|Eg,Di)\cdot P(Mb|Di)\cdot P(KB|KH)\cdot P(GA|KH,Mb)$$

In [ ]:
# ============================================================
# SLAYTTA VERİLEN KALP HASTALIĞI BAYES AĞI
# Koşullu Olasılık Tabloları (CPT) + Çıkarım
# ============================================================

print('=== KALP HASTALIĞI BAYES AĞI ===')
print('Yapı: Egzersiz(Eg), Diyet(Di) → Kalp Hastalığı(KH)')
print('      Diyet(Di)  → Mide Yanması(Mb)')
print('      Kalp Hastalığı(KH) → Kan Basıncı(KB)')
print('      Kalp Hastalığı(KH) + Mide Yanması(Mb) → Göğüs Ağrısı(GA)')

# =================================
# ÖNSEL OLASILIKLAR (kök düğümler)
# =================================
P_Eg = {True: 0.55, False: 0.45}           # Düzenli egzersiz yapıyor mu?
P_Di = {True: 0.40, False: 0.60}           # Sağlıklı diyet yapıyor mu?

# =================================
# KOŞULLU OLASILIK TABLOLARI (CPT)
# =================================
# P(KH | Eg, Di)
P_KH_gEg_Di = {
    (True,  True):  0.10,   # egzersiz + diyet → düşük risk
    (True,  False): 0.25,   # egzersiz ama kötü diyet
    (False, True):  0.35,   # iyi diyet ama egzersiz yok
    (False, False): 0.60,   # ne egzersiz ne diyet → yüksek risk
}

# P(Mb | Di)
P_Mb_gDi = {True: 0.20, False: 0.50}       # sağlıklı diyet → daha az mide yanması

# P(KB | KH) — KB: yüksek kan basıncı
P_KB_gKH = {True: 0.75, False: 0.15}

# P(GA | KH, Mb)
P_GA_gKH_Mb = {
    (True,  True):  0.95,   # KH + mide yanması → göğüs ağrısı çok olası
    (True,  False): 0.80,
    (False, True):  0.40,
    (False, False): 0.05,
}

# =================================
# ORTAK OLASILIĞI HESAPLA
# P(X) = P(Eg)·P(Di)·P(KH|Eg,Di)·P(Mb|Di)·P(KB|KH)·P(GA|KH,Mb)
# =================================
def hesapla_ortak(eg, di, kh, mb, kb, ga):
    p_eg = P_Eg[eg]
    p_di = P_Di[di]
    p_kh = P_KH_gEg_Di[(eg, di)] if kh else 1 - P_KH_gEg_Di[(eg, di)]
    p_mb = P_Mb_gDi[di]          if mb else 1 - P_Mb_gDi[di]
    p_kb = P_KB_gKH[kh]          if kb else 1 - P_KB_gKH[kh]
    p_ga = P_GA_gKH_Mb[(kh, mb)] if ga else 1 - P_GA_gKH_Mb[(kh, mb)]
    return p_eg * p_di * p_kh * p_mb * p_kb * p_ga

# Soru: Gözlem: KB=True, GA=True → P(KH=True | KB=True, GA=True) = ?
print('\n📊 ÇIKARIM: P(KH | KB=True, GA=True)')
print('Tüm kombinasyonlar üzerinden marginalleştirme...')

pay, payda = 0.0, 0.0
for eg in [True, False]:
    for di in [True, False]:
        for mb in [True, False]:
            p_kh_true  = hesapla_ortak(eg, di, True,  mb, True, True)
            p_kh_false = hesapla_ortak(eg, di, False, mb, True, True)
            pay   += p_kh_true
            payda += p_kh_true + p_kh_false

print(f'\n  P(KH=True  | KB=True, GA=True) = {pay/payda:.4f} ({pay/payda:.1%})')
print(f'  P(KH=False | KB=True, GA=True) = {1-pay/payda:.4f} ({1-pay/payda:.1%})')
print(f'\n  → "KB yüksek ve göğüs ağrısı varsa kalp hastalığı olasılığı %{pay/payda*100:.1f}"')

In [ ]:
# ============================================================
# BAYES AĞI — DAG YAPISI GÖRSELLEŞTİRME (networkx)
# ============================================================
try:
    import networkx as nx
    NETWORKX = True
except ImportError:
    NETWORKX = False

if NETWORKX:
    G = nx.DiGraph()
    kenarlar = [('Egzersiz', 'Kalp Hastalığı'),
                ('Diyet',    'Kalp Hastalığı'),
                ('Diyet',    'Mide Yanması'),
                ('Kalp Hastalığı', 'Kan Basıncı'),
                ('Kalp Hastalığı', 'Göğüs Ağrısı'),
                ('Mide Yanması',   'Göğüs Ağrısı')]
    G.add_edges_from(kenarlar)

    pos = {'Egzersiz': (0, 2), 'Diyet': (3, 2),
           'Kalp Hastalığı': (1.2, 1), 'Mide Yanması': (3, 1),
           'Kan Basıncı': (0, 0), 'Göğüs Ağrısı': (2.2, 0)}

    kok_dugumler = ['Egzersiz', 'Diyet']
    ara_dugumler = ['Kalp Hastalığı', 'Mide Yanması']
    alt_dugumler = ['Kan Basıncı', 'Göğüs Ağrısı']

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title('Kalp Hastalığı Bayes Ağı (DAG)', fontsize=13, fontweight='bold')

    nx.draw_networkx_nodes(G, pos, nodelist=kok_dugumler, node_color='#2ecc71', node_size=2500, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=ara_dugumler, node_color='#e74c3c', node_size=2500, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=alt_dugumler, node_color='#3498db', node_size=2500, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20,
                           edge_color='#2c3e50', width=2,
                           connectionstyle='arc3,rad=0.05', ax=ax)

    # Efsane
    patches = [mpatches.Patch(color='#2ecc71', label='Kök (Prior)'),
               mpatches.Patch(color='#e74c3c', label='Ara Değişken'),
               mpatches.Patch(color='#3498db', label='Yaprak (Gözlemli)')]
    ax.legend(handles=patches, loc='lower left', fontsize=9)

    ax.text(1.5, -0.5,
            'P(X) = P(Eg)·P(Di)·P(KH|Eg,Di)·P(Mb|Di)·P(KB|KH)·P(GA|KH,Mb)',
            ha='center', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('NetworkX kurulu değil — !pip install networkx ile kurabilirsiniz')
    print('\nKalp Hastalığı DAG Yapısı:')
    print('Egzersiz ──┐')
    print('           ├──▶ Kalp Hastalığı ──▶ Kan Basıncı')
    print('Diyet ─────┘       └─────────────▶ Göğüs Ağrısı')
    print('  └──▶ Mide Yanması ─────────────▶ Göğüs Ağrısı')

# Yerel Markov Özelliği gösterimi
print('\n=== YEREL MARKOV ÖZELLİĞİ ===')
print('Kalp Hastalığı düğümü için:')
print('  pa(KH) = {Egzersiz, Diyet}')
print('  NonDesc(KH) = {}')
print('  → KH ⊥ {} | {Egzersiz, Diyet}')
print()
print('Kan Basıncı düğümü için:')
print('  pa(KB) = {Kalp Hastalığı}')
print('  NonDesc(KB) = {Egzersiz, Diyet, Mide Yanması}')
print('  → KB ⊥ {Eg, Di, Mb} | {Kalp Hastalığı}')

---
# BÖLÜM 4: Topluluk Öğrenmesi

## 📖 Teorik Özet

### Binom Hata Analizi (Slaytta Verilen)
$$e_{ens} = \sum_{i=13}^{25} \binom{25}{i} \varepsilon^i (1-\varepsilon)^{25-i} \approx 0.06 \ll \varepsilon = 0.35$$

### Bias-Variance Ayrıştırması
$$\text{gen.hata}(m) = \underbrace{c_1 \cdot \text{gürültü}}_{\text{kaçınılamaz}} + \underbrace{\text{yanlılık}(m)}_{\text{bias}} + \underbrace{c_2 \cdot \text{varyans}(m)}_{\text{variance}}$$

### AdaBoost — Ağırlık Güncelleme
$$\alpha_t = \frac{1}{2}\ln\frac{1-\varepsilon_t}{\varepsilon_t} \qquad w_i^{(t+1)} = \frac{w_i^{(t)}}{Z_t} \exp(-\alpha_t y_i C_t(x_i))$$

In [ ]:
# ============================================================
# BİNOM HATA ANALİZİ — TOPLULUĞUN GÜCÜ
# ============================================================
from scipy.special import comb

def ensemble_hata(eps, n):
    """n bağımsız sınıflandırıcı, her biri eps hata oranında"""
    cogu = n // 2 + 1
    return sum(comb(n, i, exact=True) * eps**i * (1-eps)**(n-i)
               for i in range(cogu, n+1))

print('=== BİNOM HATA ANALİZİ ===')
eps_test = 0.35
n_test   = 25
e_ens    = ensemble_hata(eps_test, n_test)
print(f'ε = {eps_test}, n = {n_test}')
print(f'e_ens = {e_ens:.4f}  (Slaytta: ≈ 0.06)')
print(f'Azalma: {eps_test} → {e_ens:.4f} = %{(1-e_ens/eps_test)*100:.1f} iyileşme')

# Farklı epsilon ve n değerleri
eps_degerler = np.linspace(0.05, 0.49, 100)
n_degerler   = [5, 11, 25, 51, 101]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Binom Hata Analizi — Topluluk Öğrenmesinin Gücü', fontsize=13, fontweight='bold')

ax1 = axes[0]
for n in n_degerler:
    hatalar = [ensemble_hata(eps, n) for eps in eps_degerler]
    ax1.plot(eps_degerler, hatalar, linewidth=2, label=f'n={n}')
ax1.plot(eps_degerler, eps_degerler, 'k--', linewidth=2, label='Tek Sınıflandırıcı (ε)')
ax1.axvline(0.35, color='gray', linestyle=':', alpha=0.7)
ax1.axhline(0.06, color='gray', linestyle=':', alpha=0.7)
ax1.scatter([0.35], [e_ens], c='red', s=150, zorder=5,
            label=f'Slaytta: n=25, ε=0.35 → {e_ens:.3f}')
ax1.set_xlabel('Bireysel Hata (ε)'); ax1.set_ylabel('Topluluk Hatası')
ax1.set_title('Topluluk Boyutunun Etkisi'); ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Bias-Variance görsel
ax2 = axes[1]
karmasiklik = np.linspace(0.1, 10, 200)
bias     = 1.0 / (karmasiklik**1.2)
variance = 0.02 * karmasiklik**1.5
toplam   = bias + variance
gurultu  = 0.1 * np.ones_like(karmasiklik)

ax2.plot(karmasiklik, bias,     'b-',  lw=2.5, label='Yanlılık (Bias)')
ax2.plot(karmasiklik, variance, color='orange', lw=2.5, label='Varyans (Variance)')
ax2.plot(karmasiklik, toplam,   'g-',  lw=2.5, label='Toplam Hata')
ax2.axhline(0.1, color='red', linestyle='--', lw=1.5, label='Gürültü (Kaçınılamaz)')

opt_idx = np.argmin(toplam)
ax2.axvline(karmasiklik[opt_idx], color='purple', linestyle=':', lw=2,
            label=f'Optimum ≈ {karmasiklik[opt_idx]:.1f}')
ax2.scatter([karmasiklik[opt_idx]], [toplam[opt_idx]], c='purple', s=200, zorder=5)

ax2.set_xlabel('Model Karmaşıklığı'); ax2.set_ylabel('Hata')
ax2.set_title('Bias–Variance Trade-off'); ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.5)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ADABOOST — ADIM ADIM SLAYTTA VERİLEN MANTIK
# ============================================================

class AdaBoostManuel:
    """Slaytta anlatılan AdaBoost algoritmasının sıfırdan uygulaması"""
    
    def __init__(self, n_estimators=10):
        self.T = n_estimators
        self.alphas     = []
        self.stumps     = []
        self.gecmis     = []
    
    def fit(self, X, y):
        N = len(y)
        w = np.ones(N) / N    # Adım 1: Eşit ağırlık başlangıcı
        
        for t in range(self.T):
            # Adım 2: Ağırlıklı karar kütüğü eğit
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)
            
            # Ağırlıklı hata oranı
            yanlis = (y_pred != y).astype(float)
            eps_t = np.dot(w, yanlis)   # ε_t = Σ wⱼ·I(Cₜ≠yⱼ)
            
            if eps_t >= 0.5:
                print(f'  Tur {t+1}: ε={eps_t:.3f} ≥ 0.5 → DUR!')
                break
            
            # Alpha: α_t = ½ ln((1-ε)/ε)
            alpha_t = 0.5 * np.log((1 - eps_t) / eps_t)
            
            # Ağırlık güncelle: wᵢ ← wᵢ·exp(-αₜ·yᵢ·Cₜ(xᵢ))
            w = w * np.exp(-alpha_t * y * y_pred)
            w /= w.sum()   # normalize (Zₜ)
            
            self.alphas.append(alpha_t)
            self.stumps.append(stump)
            self.gecmis.append({'tur': t+1, 'eps': eps_t, 'alpha': alpha_t,
                                'acc': accuracy_score(y, y_pred)})
        
        return self
    
    def predict(self, X):
        # C*(x) = sign(Σ αₜ·Cₜ(x))
        toplam = np.zeros(len(X))
        for alpha, stump in zip(self.alphas, self.stumps):
            toplam += alpha * stump.predict(X)
        return np.sign(toplam)


# Uygulama
np.random.seed(42)
X_ada, y_ada = make_classification(n_samples=200, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)
y_ada = np.where(y_ada == 0, -1, 1)
X_ada_tr, X_ada_te, y_ada_tr, y_ada_te = train_test_split(X_ada, y_ada, test_size=0.3, random_state=42)

ada_manuel = AdaBoostManuel(n_estimators=15)
ada_manuel.fit(X_ada_tr, y_ada_tr)

print('=== ADABOOST — TUR BAZLI ANALİZ ===')
print("{:<6} {:<12} {:<12} {}".format("Tur", "Hata (ε)", "Alpha (α)", "Stump Acc"))
print('─' * 45)
for h in ada_manuel.gecmis:
    print(f"{h['tur']:<6} {h['eps']:<12.4f} {h['alpha']:<12.4f} {h['acc']:.4f}")

# Doğruluk hesapla
y_tahmin = ada_manuel.predict(X_ada_te)
acc_manuel = accuracy_score(y_ada_te, y_tahmin)
print(f'\n🎯 AdaBoost Manuel Test Doğruluğu: {acc_manuel:.4f}')

# sklearn AdaBoost karşılaştırma
ada_sklearn = AdaBoostClassifier(n_estimators=50, random_state=42)
ada_sklearn.fit(X_ada_tr, y_ada_tr)
print(f'🎯 sklearn AdaBoost  Test Doğruluğu: {accuracy_score(y_ada_te, ada_sklearn.predict(X_ada_te)):.4f}')

In [ ]:
# ============================================================
# BAGGING vs BOOSTING vs SINGLE — KAPSAMLI KARŞILAŞTIRMA
# ============================================================
print('=== BAGGING vs BOOSTING vs TEK MODEL ===')
print('Veri: make_classification (500 örnek, 2 özellik)')
print()

X_ens, y_ens = make_classification(n_samples=500, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)
X_ens_tr, X_ens_te, y_ens_tr, y_ens_te = train_test_split(
    X_ens, y_ens, test_size=0.25, random_state=42)

modeller = {
    'Karar Kütüğü (Stump)':    DecisionTreeClassifier(max_depth=1),
    'Derin Karar Ağacı':       DecisionTreeClassifier(max_depth=None),
    'Bagging (k=10)':          BaggingClassifier(n_estimators=10, random_state=42),
    'Bagging (k=50)':          BaggingClassifier(n_estimators=50, random_state=42),
    'Random Forest (k=50)':    RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost (k=50)':         AdaBoostClassifier(n_estimators=50, random_state=42),
}

print("{:<30} {:<14} {:<12} {}".format("Model", "Eğitim Acc", "Test Acc", "CV Acc (5-fold)"))
print('─' * 72)

sonuclar_ens = []
for isim, model in modeller.items():
    model.fit(X_ens_tr, y_ens_tr)
    train_acc = accuracy_score(y_ens_tr, model.predict(X_ens_tr))
    test_acc  = accuracy_score(y_ens_te, model.predict(X_ens_te))
    cv_acc    = cross_val_score(model, X_ens, y_ens, cv=5).mean()
    sonuclar_ens.append({'Model': isim, 'Eğitim': train_acc,
                         'Test': test_acc, 'CV': cv_acc})
    print(f'{isim:<30} {train_acc:.4f}         {test_acc:.4f}       {cv_acc:.4f}')

# Görsel
df_ens = pd.DataFrame(sonuclar_ens)
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_ens))
w = 0.28
ax.bar(x-w,   df_ens['Eğitim'], w, label='Eğitim', color='#3498db', alpha=0.85)
ax.bar(x,     df_ens['Test'],   w, label='Test',   color='#e74c3c', alpha=0.85)
ax.bar(x+w,   df_ens['CV'],     w, label='CV',     color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_ens['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Doğruluk'); ax.set_ylim(0.7, 1.05)
ax.set_title('Topluluk Yöntemleri Karşılaştırması', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
ax.axvline(1.5, color='gray', linestyle=':', alpha=0.5)
ax.text(0.5, 0.72, 'Tek Modeller', ha='center', fontsize=9, color='gray')
ax.text(3.5, 0.72, 'Topluluk Modelleri', ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

---
# BÖLÜM 5: Sınıf Dengesizliği

## 📖 Teorik Özet

**Doğruluk Paradoksu:** %1 pozitif veri → "Her şey negatif" modeli → **%99 doğruluk ama hiçbir pozitif yakalanmaz!**

**SMOTE Algoritması:** 
$$x_{yeni} = x + \lambda(x_k - x), \quad \lambda \in [0,1]$$

In [ ]:
# ============================================================
# DOĞRULUK PARADOKSU — CANLI GÖSTERİM
# ============================================================
from sklearn.dummy import DummyClassifier

# Dengesiz veri seti oluştur (1000 negatif, 50 pozitif)
np.random.seed(42)
n_neg, n_pos = 1000, 50
X_den = np.r_[np.random.randn(n_neg, 2),
              np.random.randn(n_pos, 2) + [2.5, 2.5]]
y_den = np.r_[np.zeros(n_neg), np.ones(n_pos)]

print('=== DOĞRULUK PARADOKSU ===')
print(f'Negatif (0): {n_neg} örnek | Pozitif (1): {n_pos} örnek')
print(f'Sınıf oranı: {n_pos/(n_neg+n_pos)*100:.1f}% pozitif')

X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_den, y_den, test_size=0.2, random_state=42, stratify=y_den)

# 1. Hep negatif diyen model
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_d_tr, y_d_tr)
y_pred_dummy = dummy.predict(X_d_te)

# 2. Gerçek model (eğitimsiz)
rf_den = RandomForestClassifier(n_estimators=50, random_state=42)
rf_den.fit(X_d_tr, y_d_tr)
y_pred_rf = rf_den.predict(X_d_te)

for isim, y_pred in [('Hep Negatif (Trivial)', y_pred_dummy),
                      ('Random Forest', y_pred_rf)]:
    acc = accuracy_score(y_d_te, y_pred)
    tp  = np.sum((y_d_te==1) & (y_pred==1))
    fn  = np.sum((y_d_te==1) & (y_pred==0))
    fp  = np.sum((y_d_te==0) & (y_pred==1))
    tn  = np.sum((y_d_te==0) & (y_pred==0))
    tpr = tp / (tp + fn) if (tp+fn) > 0 else 0
    print(f'\n🔹 {isim}')
    print(f'   Doğruluk: {acc:.4f} ({acc:.1%}) ← YANILTICI!')
    print(f'   TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    print(f'   Duyarlılık (Recall): {tpr:.4f}  ← GERÇEK PERFORMANS')

print('\n⚠️  "Hep negatif" model %95+ doğruluk elde ediyor ama TP=0!')

In [ ]:
# ============================================================
# SMOTE — SIFIRDAN UYGULAMA
# Slaytta verilen: x_yeni = x + λ(xk - x), λ ∈ [0,1]
# ============================================================

def smote_manuel(X_azinlik, k=5, n_yeni=50):
    """Slayttaki SMOTE algoritmasının sıfırdan uygulaması"""
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1)
    nn.fit(X_azinlik)
    
    sentetikler = []
    for _ in range(n_yeni):
        # Rastgele bir azınlık örneği seç
        idx = np.random.randint(0, len(X_azinlik))
        x   = X_azinlik[idx]
        
        # k en yakın komşuyu bul
        _, komsular = nn.kneighbors([x])
        komsu_idx = np.random.choice(komsular[0][1:])  # kendisi hariç
        x_k = X_azinlik[komsu_idx]
        
        # Interpolasyon: x_yeni = x + λ(xk - x)
        lam = np.random.random()
        x_yeni = x + lam * (x_k - x)
        sentetikler.append(x_yeni)
    
    return np.array(sentetikler)

# Azınlık örnekleri
np.random.seed(42)
X_azinlik = np.random.randn(20, 2) * 0.5 + [1.5, 1.5]
X_cogünluk = np.random.randn(200, 2)

# SMOTE uygula
X_sentetik = smote_manuel(X_azinlik, k=5, n_yeni=80)

# Görselleştir
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SMOTE: Sentetik Azınlık Artırma Tekniği', fontsize=13, fontweight='bold')

# Önce
axes[0].scatter(X_cogünluk[:,0], X_cogünluk[:,1], c='#3498db', s=20, alpha=0.5, label=f'Negatif ({len(X_cogünluk)})')
axes[0].scatter(X_azinlik[:,0],  X_azinlik[:,1],  c='#e74c3c', s=80, label=f'Pozitif ({len(X_azinlik)})')
axes[0].set_title('Önce: Dengesiz Veri')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# SMOTE
axes[1].scatter(X_azinlik[:,0],  X_azinlik[:,1],  c='#e74c3c', s=80, label='Orijinal (+)')
axes[1].scatter(X_sentetik[:,0], X_sentetik[:,1], c='#e67e22', s=40, marker='^',
                label=f'Sentetik ({len(X_sentetik)})')

# Örnek interpolasyon göster
x_ornek = X_azinlik[3]
x_komsu = X_azinlik[7]
for lam_ex in [0.25, 0.5, 0.75]:
    x_new = x_ornek + lam_ex * (x_komsu - x_ornek)
    axes[1].scatter(*x_new, c='yellow', s=120, marker='*', edgecolors='black', zorder=8)
axes[1].annotate('', xy=x_komsu, xytext=x_ornek,
                 arrowprops=dict(arrowstyle='->', color='black', lw=2))
axes[1].text((x_ornek[0]+x_komsu[0])/2, (x_ornek[1]+x_komsu[1])/2+0.1,
             'λ=0.25, 0.5, 0.75\n★=Sentetik', fontsize=7, ha='center')

axes[1].set_title('SMOTE: İnterpolasyon ile Örnekleme')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Sonra
X_pos_toplam = np.r_[X_azinlik, X_sentetik]
axes[2].scatter(X_cogünluk[:,0],   X_cogünluk[:,1],   c='#3498db', s=20, alpha=0.5, label=f'Negatif ({len(X_cogünluk)})')
axes[2].scatter(X_pos_toplam[:,0], X_pos_toplam[:,1], c='#e74c3c', s=40,
                label=f'Pozitif ({len(X_pos_toplam)})', alpha=0.7)
axes[2].set_title(f'Sonra: Dengeli Veri ({len(X_pos_toplam)}/{len(X_cogünluk)})')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 SMOTE: Orijinal pozitif örneklerin konveks zarfı içinde örnek üretir')
print('💡 Sınır bölgesine yakın örnekler için dikkatli kullanılmalıdır!')

In [ ]:
# ============================================================
# DENGESİZLİK ÇÖZÜM YÖNTEMLERİ KARŞILAŞTIRMASI
# ============================================================
from sklearn.utils import resample

print('=== DENGESİZLİK ÇÖZÜM YÖNTEMLERİ KARŞILAŞTIRMASI ===')
print(f'Orijinal: Negatif={n_neg}, Pozitif={n_pos}')

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_den, y_den, test_size=0.25, random_state=42, stratify=y_den)

# 1. Undersampling
idx_neg = np.where(y_tr2==0)[0]
idx_pos = np.where(y_tr2==1)[0]
idx_neg_under = np.random.choice(idx_neg, size=len(idx_pos)*2, replace=False)
idx_under = np.r_[idx_neg_under, idx_pos]
X_under, y_under = X_tr2[idx_under], y_tr2[idx_under]

# 2. Oversampling (Kopyalama)
n_oversample = len(idx_neg)
idx_pos_over = np.random.choice(idx_pos, size=n_oversample, replace=True)
idx_over = np.r_[idx_neg, idx_pos_over]
X_over, y_over = X_tr2[idx_over], y_tr2[idx_over]

# 3. SMOTE
X_pos_train = X_tr2[idx_pos]
X_smote_new = smote_manuel(X_pos_train, k=3, n_yeni=len(idx_neg)-len(idx_pos))
X_smote = np.r_[X_tr2, X_smote_new]
y_smote = np.r_[y_tr2, np.ones(len(X_smote_new))]

# 4. Class weight
cw_ratio = len(idx_neg) / len(idx_pos)

# Tüm yöntemleri test et
yontemler = [
    ('Orijinal (Dengesiz)',    X_tr2,   y_tr2,   {}),
    ('Undersampling',          X_under, y_under, {}),
    ('Oversampling (Kopyala)', X_over,  y_over,  {}),
    ('SMOTE',                  X_smote, y_smote, {}),
    ('Class Weight',           X_tr2,   y_tr2,   {'class_weight': {0:1, 1:cw_ratio}}),
]

print("\n{:<28} {:<8} {:<12} {:<10} {}".format("Yöntem", "Acc", "Precision", "Recall", "F1"))
print('─' * 65)

for isim, X_t, y_t, params in yontemler:
    rf = RandomForestClassifier(n_estimators=50, random_state=42, **params)
    rf.fit(X_t, y_t)
    y_p = rf.predict(X_te2)
    acc = accuracy_score(y_te2, y_p)
    prec = precision_score(y_te2, y_p, zero_division=0)
    rec  = recall_score(y_te2, y_p)
    f1   = f1_score(y_te2, y_p)
    print(f'{isim:<28} {acc:.3f}    {prec:.3f}        {rec:.3f}      {f1:.3f}')

---
# BÖLÜM 6: Performans Metrikleri

## 📖 Teorik Özet

**Confusion Matrix:** TP, TN, FP, FN

$$\text{Precision} = \frac{TP}{TP+FP}, \quad \text{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = \frac{2 \cdot r \cdot p}{r+p}, \quad F_\beta = \frac{(\beta^2+1)rp}{\beta^2 p + r}$$

$$\text{TPR} = \frac{TP}{TP+FN}, \quad \text{FPR} = \frac{FP}{FP+TN}$$

In [ ]:
# ============================================================
# SLAYTTA VERİLEN DOĞRULUK PARADOKSU SAYISAL ÖRNEĞİ
# TP=0, FP=0, FN=50, TN=4950 → %99 doğruluk ama sıfır yakalama
# ============================================================

def metrikleri_hesapla(tp, fp, fn, tn, isim='Model'):
    total = tp + fp + fn + tn
    acc   = (tp + tn) / total
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    fpr   = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr   = fn / (fn + tp) if (fn + tp) > 0 else 0
    fdr   = fp / (tp + fp) if (tp + fp) > 0 else 0
    g     = np.sqrt(prec * rec) if (prec * rec) > 0 else 0
    print(f'\n{'='*50}')
    print(f'  {isim}')
    print(f'{'='*50}')
    print(f'  Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    print(f'  ─────────────────────────────────────')
    print(f'  Doğruluk (Acc)         = {acc:.4f} ({acc:.1%})')
    print(f'  Kesinlik (Precision)   = {prec:.4f}')
    print(f'  Duyarlılık (Recall/TPR)= {rec:.4f}')
    print(f'  Özgüllük (TNR/Spec.)   = {spec:.4f}')
    print(f'  F1 Skoru               = {f1:.4f}')
    print(f'  FPR                    = {fpr:.4f}')
    print(f'  FNR                    = {fnr:.4f}')
    print(f'  FDR                    = {fdr:.4f}')
    print(f'  G-mean = √(Rec×Prec)   = {g:.4f}')

# Slayttaki Doğruluk Paradoksu örneği
metrikleri_hesapla(0, 0, 50, 4950, 'Trivial Model (Hep Negatif) — Slaytta Gösterim')

# İyi model örneği
metrikleri_hesapla(40, 15, 10, 4935, 'İyi Model (Bazı Kaçırmalar)')

# Mükemmel model
metrikleri_hesapla(50, 0, 0, 4950, 'Mükemmel Model')

# Fβ analizi
print('\n=== Fβ ANALİZİ ===')
print('Örnek: TP=40, FP=15, FN=10 için farklı β değerleri')
tp_ex, fp_ex, fn_ex = 40, 15, 10
prec_ex = tp_ex / (tp_ex + fp_ex)
rec_ex  = tp_ex / (tp_ex + fn_ex)
print(f'Precision={prec_ex:.3f}, Recall={rec_ex:.3f}')
print()
for beta in [0.5, 1.0, 2.0, 3.0]:
    f_beta = (beta**2 + 1) * prec_ex * rec_ex / (beta**2 * prec_ex + rec_ex)
    agirlik = 'Kesinliğe ağırlık' if beta<1 else ('Eşit' if beta==1 else 'Duyarlılığa ağırlık')
    print(f'  β={beta}: F_β = {f_beta:.4f} ({agirlik})')

In [ ]:
# ============================================================
# CONFUSION MATRIX — İNTERAKTİF GÖRSELLEŞTİRME
# ============================================================
from sklearn.metrics import ConfusionMatrixDisplay

# Breast Cancer veri seti üzerinde 3 model karşılaştır
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_c_tr, X_c_te, y_c_tr, y_c_te = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c)

sc = StandardScaler()
X_c_tr_s = sc.fit_transform(X_c_tr)
X_c_te_s = sc.transform(X_c_te)

modeller_cm = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':     SVC(kernel='rbf', probability=True),
    'AdaBoost':      AdaBoostClassifier(n_estimators=100, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Confusion Matrix Karşılaştırması (Breast Cancer)', fontsize=13, fontweight='bold')

for ax, (isim, model) in zip(axes, modeller_cm.items()):
    model.fit(X_c_tr_s, y_c_tr)
    y_p = model.predict(X_c_te_s)
    cm  = confusion_matrix(y_c_te, y_p)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Malign', 'Benign'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_c_te, y_p)
    f1  = f1_score(y_c_te, y_p)
    ax.set_title(f'{isim}\nAcc={acc:.3f} | F1={f1:.3f}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROC EĞRİSİ VE PR EĞRİSİ — SLAYTTAKI M1/M2 KARŞILAŞTIRMASI
# ============================================================

# 3 modelin ROC ve PR eğrileri (Breast Cancer)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ROC ve PR Eğrisi Karşılaştırması', fontsize=13, fontweight='bold')

renkler_roc = {'Random Forest': '#2ecc71', 'SVM (RBF)': '#3498db', 'AdaBoost': '#e74c3c'}

for isim, model in modeller_cm.items():
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_c_te_s)[:, 1]
    else:
        y_score = model.decision_function(X_c_te_s)
    
    # ROC
    fpr_arr, tpr_arr, _ = roc_curve(y_c_te, y_score)
    roc_auc = auc(fpr_arr, tpr_arr)
    axes[0].plot(fpr_arr, tpr_arr, lw=2, color=renkler_roc[isim],
                 label=f'{isim} (AUC={roc_auc:.3f})')
    
    # PR
    prec_arr, rec_arr, _ = precision_recall_curve(y_c_te, y_score)
    pr_auc = auc(rec_arr, prec_arr)
    axes[1].plot(rec_arr, prec_arr, lw=2, color=renkler_roc[isim],
                 label=f'{isim} (AUPRC={pr_auc:.3f})')

# ROC — referans çizgileri
axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Rastgele (AUC=0.5)')
axes[0].scatter([0],[1], c='gold', s=200, zorder=5, marker='*', label='Mükemmel (0,1)')
axes[0].set_xlabel('FPR (Yanlış Pozitif Oranı)', fontsize=11)
axes[0].set_ylabel('TPR (Duyarlılık)', fontsize=11)
axes[0].set_title('ROC Eğrisi', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([-0.02, 1.02]); axes[0].set_ylim([-0.02, 1.05])

# PR — referans çizgisi
alpha_ratio = np.sum(y_c_te==1) / len(y_c_te)
axes[1].axhline(alpha_ratio, color='gray', linestyle='--', lw=1.5,
                label=f'Rastgele (α={alpha_ratio:.2f})')
axes[1].scatter([1],[1], c='gold', s=200, zorder=5, marker='*', label='Mükemmel (1,1)')
axes[1].set_xlabel('Duyarlılık (Recall)', fontsize=11)
axes[1].set_ylabel('Kesinlik (Precision)', fontsize=11)
axes[1].set_title("PR Eğrisi\n(Dengesiz Veride ROC'dan Daha Açıklayıcı)", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-0.02, 1.02]); axes[1].set_ylim([-0.02, 1.05])

plt.tight_layout()
plt.show()

print('\n💡 ROC: TPR vs FPR — sınıf çarpıklığına görece duyarsız')
print('💡 PR : Precision vs Recall — dengesiz veride daha açıklayıcı')
print('💡 AUC ne kadar büyükse model o kadar iyi sıralama yapıyor!')

In [ ]:
# ============================================================
# OPTİMAL KARAR EŞİĞİ — SLAYTTA VERİLEN MANTIK
# Eşiği değiştirerek F1'i maksimize et
# ============================================================
rf_model = modeller_cm['Random Forest']
y_score_rf = rf_model.predict_proba(X_c_te_s)[:, 1]

esikler  = np.linspace(0.01, 0.99, 200)
f1_liste = []
acc_liste = []
prec_liste = []
rec_liste  = []

for s in esikler:
    y_th = (y_score_rf >= s).astype(int)
    f1_liste.append(f1_score(y_c_te, y_th, zero_division=0))
    acc_liste.append(accuracy_score(y_c_te, y_th))
    prec_liste.append(precision_score(y_c_te, y_th, zero_division=0))
    rec_liste.append(recall_score(y_c_te, y_th))

en_iyi_esik = esikler[np.argmax(f1_liste)]
en_iyi_f1   = max(f1_liste)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(esikler, f1_liste,   'b-', lw=2.5, label='F1')
ax.plot(esikler, acc_liste,  'g-', lw=2,   label='Doğruluk')
ax.plot(esikler, prec_liste, 'r-', lw=2,   label='Precision')
ax.plot(esikler, rec_liste,  'm-', lw=2,   label='Recall')
ax.axvline(en_iyi_esik, color='orange', linestyle='--', lw=2,
           label=f'Optimal Eşik = {en_iyi_esik:.3f}')
ax.axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Varsayılan Eşik = 0.5')
ax.scatter([en_iyi_esik], [en_iyi_f1], c='orange', s=200, zorder=6)
ax.set_xlabel('Karar Eşiği (sT)', fontsize=12)
ax.set_ylabel('Metrik Değeri', fontsize=12)
ax.set_title('Optimal Karar Eşiği Seçimi (Random Forest, Breast Cancer)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(en_iyi_esik+0.02, en_iyi_f1-0.05,
        f'F1={en_iyi_f1:.3f}', fontsize=9, color='orange')

plt.tight_layout()
plt.show()

print(f'\n📌 Varsayılan eşik (0.5): F1 = {f1_liste[np.argmin(abs(esikler-0.5))]:.4f}')
print(f'📌 Optimal eşik ({en_iyi_esik:.3f}): F1 = {en_iyi_f1:.4f}')
print(f'\n💡 Eşiği {en_iyi_esik:.3f} yaparak F1 iyileşiyor!')
print('💡 Slayttaki formül: s* = argmax_s E(s) üzerinden doğrulama kümesinde arama')

In [ ]:
# ============================================================
# MALİYET MATRİSİ — SLAYTTA VERİLEN KANSER ÖRNEĞİ
# FN maliyeti FP'den 100x daha fazla!
# ============================================================

print('=== MALİYET MATRİSİ — SLAYTTA VERİLEN KANSER ÖRNEĞİ ===')
print()
print('Maliyet Matrisi:')
print('             Tahmin +   Tahmin -')
print('  Gerçek +     C(+,+)=0   C(+,-)=100  ← Hasta kaçırılırsa')
print('  Gerçek -     C(-,+)=1   C(-,-)=0    ← Gereksiz alarm')
print()

C = {('+','+'): 0, ('+','-'): 100, ('-','+'): 1, ('-','-'): 0}

# Eşik ayarı formülü: s* = C(-,+) / (C(-,+) + C(+,-))
c_fp = C[('-','+')]
c_fn = C[('+','-')]
s_optimal_maliyet = c_fp / (c_fp + c_fn)
print(f'Optimal Eşik (Maliyet Odaklı):')
print(f's* = C(-,+) / (C(-,+) + C(+,-)) = {c_fp} / ({c_fp}+{c_fn}) = {s_optimal_maliyet:.4f}')
print(f'→ Eşiği {s_optimal_maliyet:.4f} yaparak FN hataları minimize et!')

# Beklenen maliyet karşılaştırması
print('\n=== BEKLENEN MALİYET KARŞILAŞTIRMASI ===')
for s in [0.5, s_optimal_maliyet, 0.1]:
    y_th = (y_score_rf >= s).astype(int)
    tp_m = np.sum((y_c_te==1) & (y_th==1))
    fp_m = np.sum((y_c_te==0) & (y_th==1))
    fn_m = np.sum((y_c_te==1) & (y_th==0))
    tn_m = np.sum((y_c_te==0) & (y_th==0))
    bek_maliyet = C[('+','-')]*fn_m + C[('-','+')]*fp_m
    print(f'  s={s:.4f}: TP={tp_m}, FP={fp_m}, FN={fn_m}, TN={tn_m} '
          f'→ Maliyet = {c_fn}×{fn_m} + {c_fp}×{fp_m} = {bek_maliyet}')

print('\n💡 Maliyet duyarlı öğrenme: FN/FP dengesini bilinçli ayarlamak!')
print('💡 Kanser teşhisi: FN>FP → eşiği düşür → daha fazla hasta yakala')

---
# 📝 DERS ÖZETİ — Tüm Konuların Karşılaştırması

## Algoritma Seçim Rehberi

| Durum | Önerilen Yöntem | Neden? |
|-------|-----------------|--------|
| Veri gürültülü | **Bagging / Random Forest** | Varyansı düşürür |
| Model basit, yüksek bias | **Boosting (AdaBoost)** | Biası düşürür |
| Hız + paralellik | **Bagging** | Bağımsız eğitim |
| En yüksek doğruluk | **Gradient Boosting** | Matematiksel optimizasyon |
| Sınıf dengesizliği | **SMOTE + Maliyet Ağırlığı** | Azınlığı artır |
| Değişkenler arası bağımlılık | **Bayes Ağı** | DAG ile modelleme |
| Doğrusal olmayan sınır | **SVM (RBF Kernel) / MLP** | Kernel trick |

## Metrik Seçim Rehberi

| Durum | Metrik |
|-------|--------|
| Dengeli veri | Accuracy, F1 |
| Dengesiz veri | **PR-AUC**, Recall, F_β |
| FN pahalı (kanser, fraud) | **Recall (TPR)**, F₂ |
| FP pahalı (spam filtresi) | **Precision**, F₀.₅ |
| Model sıralama kalitesi | **ROC-AUC** |

In [ ]:
# ============================================================
# KAPSAMLI KARŞILAŞTIRMA: Tüm Algoritmaların Özeti
# ============================================================
cancer_data = load_breast_cancer()
X_fin, y_fin = cancer_data.data, cancer_data.target
X_f_tr, X_f_te, y_f_tr, y_f_te = train_test_split(
    X_fin, y_fin, test_size=0.2, random_state=42, stratify=y_fin)

sc_fin = StandardScaler()
X_f_tr_s = sc_fin.fit_transform(X_f_tr)
X_f_te_s = sc_fin.transform(X_f_te)

modeller_son = {
    'Karar Kütüğü':        DecisionTreeClassifier(max_depth=1),
    'SVM (Lineer)':        SVC(kernel='linear', probability=True),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True),
    'MLP (100,50)':        MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500, random_state=42),
    'Naive Bayes':         GaussianNB(),
    'Bagging (k=50)':      BaggingClassifier(n_estimators=50, random_state=42),
    'Random Forest (50)':  RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost (50)':       AdaBoostClassifier(n_estimators=50, random_state=42),
}

print('=== TÜM ALGORİTMALARIN KAPSAMLI KARŞILAŞTIRMASI ===')
print('Veri Seti: Breast Cancer (569 örnek, 30 özellik, 2 sınıf)')
print()
print("{:<25} {:<8} {:<8} {:<8} {:<8} {}".format("Model", "Acc", "Prec", "Rec", "F1", "ROC-AUC"))
print('─' * 65)

kayitlar = []
for isim, model in modeller_son.items():
    model.fit(X_f_tr_s, y_f_tr)
    y_p  = model.predict(X_f_te_s)
    acc  = accuracy_score(y_f_te, y_p)
    prec = precision_score(y_f_te, y_p)
    rec  = recall_score(y_f_te, y_p)
    f1v  = f1_score(y_f_te, y_p)
    if hasattr(model, 'predict_proba'):
        fpr_r, tpr_r, _ = roc_curve(y_f_te, model.predict_proba(X_f_te_s)[:,1])
        roc_a = auc(fpr_r, tpr_r)
    else:
        roc_a = float('nan')
    kayitlar.append({'Model':isim,'Acc':acc,'Prec':prec,'Rec':rec,'F1':f1v,'AUC':roc_a})
    print(f'{isim:<25} {acc:.3f}    {prec:.3f}    {rec:.3f}    {f1v:.3f}    {roc_a:.3f}')

df_son = pd.DataFrame(kayitlar)
en_iyi = df_son.nlargest(1, 'F1').iloc[0]
print(f'\n🏆 En İyi Model (F1 bazında): {en_iyi["Model"]} (F1={en_iyi["F1"]:.4f})')